# Integração da base de alunos com o Atlas do Desenvolvimento Humano

Este notebook realiza o enriquecimento da base de alunos por meio da
integração com indicadores municipais provenientes do Atlas do
Desenvolvimento Humano no Brasil (ADH).

O Atlas disponibiliza o Índice de Desenvolvimento Humano Municipal
(IDHM) e indicadores relacionados a dimensões como demografia,
educação, renda, trabalho, habitação e vulnerabilidade social dos
municípios brasileiros.

Os dados utilizados foram obtidos por meio da plataforma
[Base dos Dados](https://basedosdados.org/dataset/cbfc7253-089b-44e2-8825-755e1419efc8?table=ec5fb3d1-fa98-4ab3-8a02-4b9950048a83),
na tabela `mundo_onu_adh.municipio`. O dicionário de dados utilizado
como referência para interpretação das variáveis foi preservado na
documentação do projeto.

A fonte utilizada possui cobertura temporal de 1991 a 2010. Dessa
forma, os indicadores disponíveis não representam o mesmo período da
avaliação educacional analisada no projeto. Essa limitação temporal
será considerada na utilização e interpretação das informações
incorporadas à base analítica.

O processo de integração será conduzido de forma controlada, com
auditoria prévia da fonte externa e validação posterior do merge,
buscando preservar a granularidade e a consistência da base de alunos.

## 1. Carregamento das bases

Nesta etapa serão carregadas a base de alunos da camada Gold e a base
do Atlas do Desenvolvimento Humano, que serão utilizadas no processo
de integração.

In [0]:
# Objetivo:
#
# Carregar as bases de alunos e do Atlas
# do Desenvolvimento Humano.
#
# Justificativa:
#
# O carregamento separado das fontes permite
# verificar suas dimensões e estruturas antes
# de realizar qualquer operação de integração.
#
# Ação:
#
# Importa a biblioteca Pandas, realiza a
# leitura dos arquivos e armazena os dados
# em DataFrames distintos.

import pandas as pd

df_alunos = pd.read_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/alunos/alunos_gold.csv",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

df_atlas = pd.read_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/fontes_externas/atlas_desenvolvimento_humano/mundo_onu_adh_municipio.csv",
    sep=",",
    encoding="utf-8-sig",
    low_memory=False
)

In [0]:
# Objetivo:
#
# Confirmar o carregamento das bases utilizadas
# no processo de integração.
#
# Justificativa:
#
# A verificação das dimensões permite registrar
# o volume inicial de dados de cada fonte e
# identificar eventuais problemas de leitura
# antes do início da auditoria.
#
# Ação:
#
# Exibe a quantidade de linhas e colunas
# presentes em cada DataFrame.

print("Base de alunos:", df_alunos.shape)
print("Base Atlas:", df_atlas.shape)

## 2. Auditoria da base Atlas

Nesta etapa será analisada a estrutura da base do Atlas do
Desenvolvimento Humano, com atenção à sua granularidade, cobertura
temporal, chaves de identificação municipal e consistência dos dados.

A auditoria fornecerá os elementos necessários para definir de forma
segura a estratégia de integração com a base de alunos.

In [0]:
# Objetivo:
#
# Analisar a cobertura temporal da base do
# Atlas do Desenvolvimento Humano.
#
# Justificativa:
#
# A identificação dos períodos disponíveis e
# da quantidade de municípios em cada ano é
# necessária para definir o recorte temporal
# mais adequado para a integração.
#
# Ação:
#
# Agrupa os dados por ano e calcula a quantidade
# de registros e de municípios distintos em
# cada período.

auditoria_temporal = (
    df_atlas
    .groupby("ano")
    .agg(
        registros=("id_municipio", "size"),
        municipios=("id_municipio", "nunique")
    )
)

auditoria_temporal

### Definição do recorte temporal

A auditoria identificou registros para os anos de 1991, 2000 e 2010,
com 5.565 municípios representados em cada período.

Para o enriquecimento da base de alunos, será utilizado o ano de 2010,
por corresponder ao período mais recente disponível no Atlas do
Desenvolvimento Humano.

A diferença temporal em relação à avaliação educacional de 2025
constitui uma limitação da fonte e deverá ser considerada na
interpretação dos indicadores incorporados.

In [0]:
# Objetivo:
#
# Selecionar o período mais recente disponível
# na base do Atlas do Desenvolvimento Humano.
#
# Justificativa:
#
# O ano de 2010 corresponde ao período mais
# recente disponível na fonte e será utilizado
# como referência para o enriquecimento da
# base de alunos.
#
# Ação:
#
# Filtra os registros referentes ao ano de
# 2010 e armazena o resultado em um novo
# DataFrame.

df_atlas_2010 = (
    df_atlas
    .loc[df_atlas["ano"] == 2010]
    .copy()
)

df_atlas_2010.shape

### Seleção dos indicadores candidatos

Com o objetivo de incorporar informações contextuais relevantes sem
aumentar excessivamente a dimensionalidade da base analítica, foram
selecionados seis indicadores candidatos do Atlas do Desenvolvimento
Humano.

A seleção foi realizada com base nas definições disponíveis no
dicionário da fonte e buscou representar dimensões distintas do
contexto socioeconômico municipal. Os indicadores serão submetidos
à auditoria antes de sua incorporação à base de alunos.

| Variável | Definição | Dimensão analítica | Justificativa |
|---|---|---|---|
| `idhm` | Índice de Desenvolvimento Humano Municipal | Desenvolvimento humano | Representar o nível geral de desenvolvimento do município. |
| `idhm_e` | Índice de Desenvolvimento Humano Municipal - Dimensão Educação | Educação | Representar a dimensão educacional do desenvolvimento humano municipal. |
| `renda_pc` | Renda per capita média | Renda | Representar o nível médio de renda da população do município. |
| `indice_gini` | Índice de Gini | Desigualdade | Representar o grau de desigualdade na distribuição de renda. |
| `prop_pobreza_criancas` | Proporção de crianças pobres | Vulnerabilidade infantil | Representar a exposição da população infantil à pobreza. |
| `taxa_criancas_dom_sem_fund` | Percentual de crianças que vivem em domicílios em que nenhum dos moradores tem o ensino fundamental completo | Contexto educacional domiciliar | Representar as condições educacionais do ambiente domiciliar das crianças. |

A variável `id_municipio` será mantida adicionalmente como chave
técnica para a integração com a base de alunos.

### Auditoria dos indicadores candidatos

Antes da incorporação dos indicadores selecionados à base de alunos,
será realizada uma auditoria de sua estrutura e qualidade.

A análise buscará verificar tipos de dados, valores ausentes,
cardinalidade e estatísticas descritivas, permitindo identificar
eventuais inconsistências ou limitações antes da integração.

In [0]:
# Objetivo:
#
# Avaliar a estrutura e a completude dos
# indicadores candidatos selecionados.
#
# Justificativa:
#
# A verificação dos tipos de dados, valores
# ausentes e cardinalidade permite identificar
# possíveis problemas antes da incorporação
# dos indicadores à base de alunos.
#
# Ação:
#
# Calcula o tipo de dado, a quantidade e o
# percentual de valores ausentes e o número
# de valores distintos de cada indicador.

indicadores_atlas = [
    "idhm",
    "idhm_e",
    "renda_pc",
    "indice_gini",
    "prop_pobreza_criancas",
    "taxa_criancas_dom_sem_fund"
]

auditoria_indicadores = pd.DataFrame({
    "tipo": df_atlas_2010[indicadores_atlas].dtypes.astype(str),
    "ausentes": df_atlas_2010[indicadores_atlas].isna().sum(),
    "percentual_ausentes": (
        df_atlas_2010[indicadores_atlas]
        .isna()
        .mean()
        .mul(100)
        .round(2)
    ),
    "valores_distintos": (
        df_atlas_2010[indicadores_atlas]
        .nunique()
    )
})

auditoria_indicadores

In [0]:
# Objetivo:
#
# Avaliar a distribuição e a plausibilidade
# dos indicadores candidatos selecionados.
#
# Justificativa:
#
# A análise das estatísticas descritivas permite
# identificar amplitudes, valores extremos e
# possíveis inconsistências antes da integração
# dos indicadores à base de alunos.
#
# Ação:
#
# Calcula estatísticas descritivas dos seis
# indicadores candidatos e organiza o resultado
# com as variáveis nas linhas.

estatisticas_indicadores = (
    df_atlas_2010[indicadores_atlas]
    .describe()
    .T
    .round(3)
)

estatisticas_indicadores

In [0]:
# Objetivo:
#
# Avaliar a relação entre os indicadores
# candidatos selecionados.
#
# Justificativa:
#
# Correlações muito elevadas entre variáveis
# podem indicar redundância de informação e
# devem ser avaliadas antes da integração
# dos indicadores à base de alunos.
#
# Ação:
#
# Calcula a matriz de correlação de Pearson
# entre os seis indicadores candidatos.

correlacao_indicadores = (
    df_atlas_2010[indicadores_atlas]
    .corr(method="pearson")
    .round(3)
)

correlacao_indicadores

### Avaliação da redundância entre os indicadores

A análise de correlação identificou relações elevadas entre alguns dos
indicadores selecionados, especialmente entre medidas relacionadas ao
desenvolvimento humano, educação, renda e vulnerabilidade social.

Apesar da possível redundância, os seis indicadores serão mantidos
nesta etapa, pois representam dimensões conceitualmente distintas e
não apresentaram problemas de completude ou plausibilidade.

A eventual exclusão de variáveis por redundância será avaliada
posteriormente, durante a análise exploratória e a modelagem, quando
também estarão disponíveis os atributos provenientes das demais fontes
que comporão a base analítica.

In [0]:
# Objetivo:
#
# Validar a chave municipal utilizada para
# integração da base do Atlas.
#
# Justificativa:
#
# A chave de integração deve estar completa e
# apresentar um único registro por município
# no recorte temporal selecionado, evitando
# multiplicação indevida de registros no merge.
#
# Ação:
#
# Verifica o tipo de dado, valores ausentes,
# quantidade de registros, municípios distintos
# e duplicidades da variável id_municipio.

auditoria_chave_atlas = pd.Series({
    "tipo": str(df_atlas_2010["id_municipio"].dtype),
    "ausentes": df_atlas_2010["id_municipio"].isna().sum(),
    "registros": len(df_atlas_2010),
    "municipios_distintos": df_atlas_2010["id_municipio"].nunique(),
    "duplicados": df_atlas_2010["id_municipio"].duplicated().sum()
})

auditoria_chave_atlas

In [0]:
# Objetivo:
#
# Verificar a compatibilidade estrutural das
# chaves municipais utilizadas na integração.
#
# Justificativa:
#
# Antes do merge, é necessário conhecer os
# tipos de dados e a ocorrência de valores
# ausentes nas chaves das duas bases para
# identificar eventual necessidade de
# padronização.
#
# Ação:
#
# Compara os tipos de dados e a quantidade
# de valores ausentes de CO_MUNICIPIO e
# id_municipio.

auditoria_chaves = pd.DataFrame({
    "base": ["Alunos", "Atlas"],
    "variavel": ["CO_MUNICIPIO", "id_municipio"],
    "tipo": [
        str(df_alunos["CO_MUNICIPIO"].dtype),
        str(df_atlas_2010["id_municipio"].dtype)
    ],
    "ausentes": [
        df_alunos["CO_MUNICIPIO"].isna().sum(),
        df_atlas_2010["id_municipio"].isna().sum()
    ]
})

auditoria_chaves

In [0]:
# Objetivo:
#
# Padronizar os tipos das chaves municipais
# utilizadas na integração das bases.
#
# Justificativa:
#
# As chaves devem possuir tipos compatíveis
# para permitir comparações e a realização
# segura do merge, preservando os valores
# ausentes já identificados na base de alunos.
#
# Ação:
#
# Converte CO_MUNICIPIO e id_municipio para
# o tipo inteiro anulável Int64 do Pandas.

df_alunos["CO_MUNICIPIO"] = (
    df_alunos["CO_MUNICIPIO"]
    .astype("Int64")
)

df_atlas_2010["id_municipio"] = (
    df_atlas_2010["id_municipio"]
    .astype("Int64")
)

df_alunos["CO_MUNICIPIO"].dtype, df_atlas_2010["id_municipio"].dtype

In [0]:
# Objetivo:
#
# Verificar a cobertura dos municípios da
# base de alunos na base do Atlas.
#
# Justificativa:
#
# Antes do merge, é necessário confirmar se
# todos os municípios com código informado
# possuem correspondência no Atlas, separando
# essa situação dos registros cuja chave
# municipal já está ausente na base de alunos.
#
# Ação:
#
# Compara os códigos municipais distintos
# das duas bases e identifica municípios
# presentes nos alunos sem correspondência
# no Atlas.

municipios_alunos = set(
    df_alunos["CO_MUNICIPIO"]
    .dropna()
    .unique()
)

municipios_atlas = set(
    df_atlas_2010["id_municipio"]
    .dropna()
    .unique()
)

municipios_sem_correspondencia = (
    municipios_alunos - municipios_atlas
)

print(
    "Municípios distintos na base de alunos:",
    len(municipios_alunos)
)

print(
    "Municípios distintos no Atlas:",
    len(municipios_atlas)
)

print(
    "Municípios dos alunos sem correspondência no Atlas:",
    len(municipios_sem_correspondencia)
)

In [0]:
# Objetivo:
#
# Identificar os municípios da base de alunos
# que não possuem correspondência no Atlas.
#
# Justificativa:
#
# A identificação dos municípios e da quantidade
# de alunos afetados permite dimensionar a
# divergência e investigar sua causa antes
# da realização do merge.
#
# Ação:
#
# Filtra os registros associados aos códigos
# municipais sem correspondência, agrupa por
# código e nome do município e contabiliza
# a quantidade de alunos afetados.

municipios_nao_encontrados = (
    df_alunos
    .loc[
        df_alunos["CO_MUNICIPIO"]
        .isin(municipios_sem_correspondencia)
    ]
    .groupby(
        ["CO_MUNICIPIO", "NO_MUNICIPIO"],
        dropna=False
    )
    .size()
    .reset_index(name="quantidade_alunos")
    .sort_values(
        "quantidade_alunos",
        ascending=False
    )
)

municipios_nao_encontrados

### Cobertura municipal do Atlas

A comparação das chaves municipais identificou seis municípios
presentes na base de alunos que não possuem correspondência no Atlas
de 2010.

A divergência decorre da diferença temporal entre as fontes. Cinco
desses municípios foram instalados após 2010 — Balneário Rincão,
Mojuí dos Campos, Pescaria Brava, Paraíso das Águas e Pinto Bandeira —
e Boa Esperança do Norte foi instalado posteriormente.

Ao todo, 820 alunos estão vinculados a esses municípios. Esses
registros serão preservados na base, permanecendo sem os indicadores
do Atlas após a integração.

Além deles, existem 510 alunos sem código municipal na própria base
de origem, situação previamente identificada e documentada.

Dessa forma, espera-se que 1.330 alunos permaneçam sem indicadores
do Atlas após o merge. Essas ausências serão preservadas nesta etapa
e reavaliadas posteriormente durante a preparação dos dados para
modelagem.

In [0]:
# Objetivo:
#
# Consolidar a cobertura esperada da integração
# entre a base de alunos e o Atlas.
#
# Justificativa:
#
# A quantificação prévia dos registros sem chave
# municipal e daqueles vinculados a municípios
# sem correspondência permite estabelecer uma
# referência para a validação posterior do merge.
#
# Ação:
#
# Calcula a quantidade de alunos sem código
# municipal, alunos vinculados aos seis municípios
# não encontrados e o total esperado de registros
# sem indicadores do Atlas.

alunos_sem_municipio = (
    df_alunos["CO_MUNICIPIO"]
    .isna()
    .sum()
)

alunos_municipios_sem_atlas = (
    df_alunos["CO_MUNICIPIO"]
    .isin(municipios_sem_correspondencia)
    .sum()
)

total_esperado_sem_atlas = (
    alunos_sem_municipio
    + alunos_municipios_sem_atlas
)

print(
    "Alunos sem código municipal:",
    alunos_sem_municipio
)

print(
    "Alunos em municípios sem correspondência no Atlas:",
    alunos_municipios_sem_atlas
)

print(
    "Total esperado sem indicadores do Atlas:",
    total_esperado_sem_atlas
)

## 3. Integração das bases

A integração será realizada utilizando a base de alunos como referência,
de forma a preservar integralmente sua população de 1.966.605 registros.

Serão incorporados do Atlas a chave municipal e os seis indicadores
selecionados e auditados na etapa anterior. A relação esperada entre
as bases é de muitos alunos para um município.

Será utilizado um `left join`, permitindo preservar também os alunos
sem correspondência municipal no Atlas, cujas causas foram previamente
identificadas e documentadas.

In [0]:
# Objetivo:
#
# Integrar os indicadores selecionados do
# Atlas à base de alunos.
#
# Justificativa:
#
# O left join preserva integralmente a população
# de alunos, enquanto a validação da cardinalidade
# garante que cada aluno encontre no máximo um
# registro municipal no Atlas.
#
# Ação:
#
# Seleciona a chave e os seis indicadores do
# Atlas e realiza o merge com a base de alunos,
# mantendo um indicador auxiliar para auditoria.

colunas_atlas = [
    "id_municipio",
    "idhm",
    "idhm_e",
    "renda_pc",
    "indice_gini",
    "prop_pobreza_criancas",
    "taxa_criancas_dom_sem_fund"
]

df_alunos_atlas = df_alunos.merge(
    df_atlas_2010[colunas_atlas],
    how="left",
    left_on="CO_MUNICIPIO",
    right_on="id_municipio",
    validate="many_to_one",
    indicator=True
)

df_alunos_atlas.shape

In [0]:
# Objetivo:
#
# Validar a correspondência dos registros
# após a integração com o Atlas.
#
# Justificativa:
#
# A distribuição do indicador de merge permite
# verificar se a quantidade de alunos sem
# correspondência coincide com a expectativa
# estabelecida antes da integração.
#
# Ação:
#
# Contabiliza os registros conforme o resultado
# da correspondência entre as duas bases.

df_alunos_atlas["_merge"].value_counts()

In [0]:
# Objetivo:
#
# Validar a origem dos registros que não
# encontraram correspondência no Atlas.
#
# Justificativa:
#
# A quantidade total de registros sem
# correspondência já coincide com a expectativa,
# mas é necessário confirmar se esses registros
# pertencem exatamente aos dois grupos
# previamente identificados.
#
# Ação:
#
# Isola os registros classificados como
# left_only e contabiliza aqueles sem código
# municipal e aqueles vinculados aos municípios
# sem correspondência no Atlas.

registros_sem_atlas = (
    df_alunos_atlas
    .loc[df_alunos_atlas["_merge"] == "left_only"]
)

sem_codigo_municipal = (
    registros_sem_atlas["CO_MUNICIPIO"]
    .isna()
    .sum()
)

municipio_sem_atlas = (
    registros_sem_atlas["CO_MUNICIPIO"]
    .isin(municipios_sem_correspondencia)
    .sum()
)

outros_casos = (
    len(registros_sem_atlas)
    - sem_codigo_municipal
    - municipio_sem_atlas
)

print(
    "Sem código municipal:",
    sem_codigo_municipal
)

print(
    "Municípios sem correspondência no Atlas:",
    municipio_sem_atlas
)

print(
    "Outros casos:",
    outros_casos
)

## 4. Validação da integração

Após a realização do merge, a base integrada será submetida a
validações de integridade e completude.

Serão verificados a preservação da população de alunos, a ocorrência
de duplicidades e o comportamento dos indicadores incorporados,
considerando as ausências previamente identificadas e documentadas.

As colunas auxiliares utilizadas durante a integração serão removidas
somente após a conclusão dessas verificações.

In [0]:
# Objetivo:
#
# Validar a integridade da população de alunos
# após a integração com o Atlas.
#
# Justificativa:
#
# A integração deve preservar a granularidade
# original da base, mantendo um único registro
# para cada aluno.
#
# Ação:
#
# Verifica a quantidade total de registros,
# alunos distintos e duplicidades de ID_ALUNO
# na base resultante do merge.

integridade_alunos = pd.Series({
    "registros": len(df_alunos_atlas),
    "alunos_distintos": df_alunos_atlas["ID_ALUNO"].nunique(),
    "id_aluno_duplicados": (
        df_alunos_atlas["ID_ALUNO"]
        .duplicated()
        .sum()
    )
})

integridade_alunos

In [0]:
# Objetivo:
#
# Validar a completude dos indicadores do
# Atlas após a integração.
#
# Justificativa:
#
# As ausências nos indicadores incorporados
# devem corresponder exclusivamente aos registros
# sem cobertura municipal previamente identificados
# antes do merge.
#
# Ação:
#
# Contabiliza os valores ausentes em cada
# indicador do Atlas e compara o resultado
# com o total esperado.

ausencias_indicadores_atlas = pd.DataFrame({
    "ausentes": (
        df_alunos_atlas[indicadores_atlas]
        .isna()
        .sum()
    )
})

ausencias_indicadores_atlas["esperado"] = (
    total_esperado_sem_atlas
)

ausencias_indicadores_atlas["conforme"] = (
    ausencias_indicadores_atlas["ausentes"]
    == ausencias_indicadores_atlas["esperado"]
)

ausencias_indicadores_atlas

In [0]:
# Objetivo:
#
# Remover as colunas auxiliares utilizadas
# durante o processo de integração.
#
# Justificativa:
#
# As variáveis id_municipio e _merge foram
# utilizadas exclusivamente como apoio ao
# merge e às validações, não sendo necessárias
# na base analítica resultante.
#
# Ação:
#
# Remove as duas colunas auxiliares e verifica
# a dimensão final da base integrada.

df_alunos_atlas = (
    df_alunos_atlas
    .drop(
        columns=[
            "id_municipio",
            "_merge"
        ]
    )
)

df_alunos_atlas.shape

## 5. Persistência da base integrada

Após a conclusão das validações, a base enriquecida com os indicadores
selecionados do Atlas será persistida para utilização na próxima etapa
de integração do projeto.

O arquivo resultante preserva a granularidade original dos alunos e
incorpora os seis indicadores municipais validados anteriormente.

In [0]:
# Objetivo:
#
# Persistir a base de alunos enriquecida
# com os indicadores selecionados do Atlas.
#
# Justificativa:
#
# A persistência da base validada permite sua
# utilização como entrada na próxima etapa de
# enriquecimento, sem necessidade de repetir
# o processamento realizado neste notebook.
#
# Ação:
#
# Salva a base integrada em formato CSV,
# preservando a codificação e o separador
# adotados no projeto.

df_alunos_atlas.to_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/alunos_atlas/alunos_atlas.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)

In [0]:
# Objetivo:
#
# Validar o arquivo persistido após a
# integração com o Atlas.
#
# Justificativa:
#
# A releitura do arquivo permite confirmar
# que a base foi gravada corretamente e
# preservou sua dimensão antes de ser utilizada
# na próxima etapa de enriquecimento.
#
# Ação:
#
# Realiza uma leitura de controle do arquivo
# persistido e verifica sua dimensão.

df_validacao = pd.read_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/alunos_atlas/alunos_atlas.csv",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

df_validacao.shape